In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import ast
import random
import os
import re

In [ ]:
def read_data(file_path, annotator_id):
    """
    Reads the annotated data and parses it into a desired structured Df.

    """
    data = []
    current_text = ""
    with open(file_path, 'r', encoding='utf-8') as file:
        for line in file:
            line = line.strip()
            if line.startswith('#Text='):
                current_text = line.split('=')[1].strip()  # Extract the text following this prefix.
            elif line and not line.startswith('#'):
                parts = line.split('\t')
                if len(parts) >= 4:
                    sentence_id, token_id = parts[0].split('-')
                    if len(sentence_id) == 0 or len(token_id) == 0:
                        continue  # Skip entries without proper ID formatting.

                    # The labels might have annotations like Object[1], split and clean them
                    spo_labels = [re.sub(r'\[\d+\]', '', label) for label in parts[3].split('|')]
                    while len(spo_labels) < 3:
                        spo_labels.append('_')

                    token_data = {
                        'annotator_id': annotator_id,
                        'sentence_id': sentence_id,
                        'token_id': token_id,
                        'span': parts[1],
                        'token': parts[2],
                        'SPO-label_1': spo_labels[0] if len(spo_labels) > 0 else '_',
                        'SPO-label_2': spo_labels[1] if len(spo_labels) > 1 else '_',
                        'SPO-label_3': spo_labels[2] if len(spo_labels) > 2 else '_',
                        'text': current_text
                    }
                    data.append(token_data)
                else:
                    print(f"Skipping malformed line in {file_path}: {line}")

    return pd.DataFrame(data)


def save_data(df, output_directory, output_file_name):
    """
    Saves Df to CSV.

    """
    # Ensures the output directory exists
    os.makedirs(output_directory, exist_ok=True)
    
    output_file_path = os.path.join(output_directory, output_file_name)
    
    df.to_csv(output_file_path, index=False, encoding='utf-8')
    print(f"Annotated data saved to: {output_file_path}")


def merge_data(file_paths):
    """
    Merges the annotated data into a single Df.

    """
    all_annotations = []
    for file_path in file_paths:
        annotations = pd.read_csv(file_path)  
        all_annotations.append(annotations)

    # Concatenate all Df's into a single Df
    combined_annotations = pd.concat(all_annotations, ignore_index=True)\

    return combined_annotations


# Process and save individual annotated data files
ANNOTATED_TSV_FILES = [
    ('conversation annotation/annotated data/annotator1_200.tsv', 'annotator_1'),
    ('conversation annotation/annotated data/annotator2_200.tsv', 'annotator_2'),
    ('conversation annotation/annotated data/annotator3_200.tsv', 'annotator_3')
]
OUTPUT_DIRECTORY = 'conversation annotation/annotated data'


for file_path, annotator_id in ANNOTATED_TSV_FILES:
    df = read_data(file_path, annotator_id)
    save_data(df, OUTPUT_DIRECTORY, f'annotated_test_data_{annotator_id}.csv')


# Merge the annotated data 
ANNOTATED_CSV_FILES = [
    ('conversation annotation/annotated data/annotated_test_data_annotator_1.csv'),
    ('conversation annotation/annotated data/annotated_test_data_annotator_2.csv'),
    ('conversation annotation/annotated data/annotated_test_data_annotator_3.csv')
]

merged_annotations = merge_data(ANNOTATED_CSV_FILES)
save_data(merged_annotations, OUTPUT_DIRECTORY, f'merged_annotated_test_data.csv')
   

In [ ]:

def aggregate_annotations(data):
    """
    Aggregates tokens and labels by 'annotator_id' and 'sentence_id'.
    
    """
    def concat_items(series):
        """
        Takes a pandas series object and converts it into a list.

        """
        return list(series)
    
    # Aggregate tokens and SPO-labels by 'annotator_id' and 'sentence_id'
    grouped = data.groupby(['annotator_id', 'sentence_id'])

    return grouped.agg({
        'token': concat_items,
        'SPO-label_1': concat_items,
        'SPO-label_2': concat_items,
        'SPO-label_3': concat_items,
        'text': 'first'
    })


def combine_labels(row):
    """
    Combines the SPO-labels into a single string for each token.
    
    """
    spo_labels = []
    for i in range(len(row['token'])):
        # Collect all SPO-labels 
        spo_labels_list = [row['SPO-label_1'][i], row['SPO-label_2'][i], row['SPO-label_3'][i]]
        # Remove placeholder '_' and keep duplicates
        spo_labels_filtered = [label for label in spo_labels_list if label != '_']
        # Join the SPO-labels with ';'
        spo_labels.append(';'.join(spo_labels_filtered))

    return spo_labels


def prepare_final_dataframe(aggregated):
    """
    Cleans and restructures the data into a Df ready for further analysis.
    
    """
    # Combines the SPO-labels and store them in a list
    spo_labels_list = aggregated.apply(combine_labels, axis=1)

    # Assign the list as a new column
    aggregated['spo_labels'] = spo_labels_list

    # Rename columns and drop unused ones
    final_data = aggregated.rename(columns={
        'token': 'tokens',
        'text': 'sentence',
    }).drop(columns=['SPO-label_1', 'SPO-label_2', 'SPO-label_3'])

    # Reset the index and keep the old index columns as regular columns
    final_data.reset_index(inplace=True, drop=False)

    # Replace empty strings in lists with 'None'
    final_data['spo_labels'] = final_data['spo_labels'].apply(
        lambda x: [tag if tag.strip() else 'None' for tag in x])

    return final_data


# Aggregate and prepare the final annotated data
aggregated = aggregate_annotations(merged_annotations)
final_annotated_data = prepare_final_dataframe(aggregated)
final_annotated_data.to_csv('conversation annotation/annotated data/final_annotated_test_data.csv', index=False)



In [ ]:
file_path = 'conversation annotation/annotated data/final_annotated_test_data.csv'
data = pd.read_csv(file_path)


def convert_to_list(row):
    """ 
    Converts string lists into actual lists
    
    """
    return ast.literal_eval(row)

# Convert the 'spo_labels' column from string representation to actual list
data['spo_labels'] = data['spo_labels'].apply(convert_to_list)
data.head()

,annotator_id,sentence_id,tokens,sentence,spo_labels
0,annotator_1,1,"['P', ':', 'I', 'have', 'been', 'facing', 'som...",P : I have been facing some discomfort in my f...,"[None, None, Subject, Predicate, Predicate, Pr..."
1,annotator_1,2,"['Is', 'this', 'connected', 'to', 'my', 'diabe...",Is this connected to my diabetes ?,"[None, Subject, Predicate, Predicate, Object, ..."
2,annotator_1,3,"['A', ':', 'It', 'could', 'be', ',', 'Abdullah...","A : It could be , Abdullah .","[None, None, None, None, None, None, None, None]"
3,annotator_1,4,"['Diabetes', 'can', 'cause', 'nerve', 'damage'...",Diabetes can cause nerve damage known as perip...,"[Subject, Predicate, Predicate, Object, Object..."
4,annotator_1,5,"['This', 'can', 'lead', 'to', 'a', 'sense', 'o...","This can lead to a sense of discomfort , numbn...","[Subject, Predicate, Predicate, Object, Object..."


In [208]:

def contingency_spo_matrix(data):
    """
    Creates a token-level contingency matrix for SPO-labels.
    
    """
    token_spo_contingency = pd.DataFrame()

    for _, row in data.iterrows():
        sentence_id = row['sentence_id']
        for token_index, spo_label in enumerate(row['spo_labels']):
            token_id = f"{sentence_id}_{token_index}"
            print(f"Debug: Processing token {token_id} with tag {spo_label}")

            # Ensure that columns for new labels are added properly
            if spo_label not in token_spo_contingency.columns:
                token_spo_contingency[spo_label] = 0  # Initialize column for new tag
                print(f"Debug: Adding new column for tag '{spo_label}'")

            # Add token_id as a new row if it doesn't exist
            if token_id not in token_spo_contingency.index:
                new_row = pd.Series(0, index=token_spo_contingency.columns, name=token_id)
                token_spo_contingency = pd.concat([token_spo_contingency, pd.DataFrame(new_row).T])
                print(f"Debug: Adding new row for token {token_id}")

            # Update the count for the current tag
            token_spo_contingency.at[token_id, spo_label] += 1
            print(f"Debug: Incrementing {token_id} for label '{spo_label}' to {token_spo_contingency.at[token_id, spo_label]}")

    token_spo_contingency.fillna(0, inplace=True)
    return token_spo_contingency


# Display the token-level contingency matrix for only SPO-tags
table = contingency_spo_matrix(data)
print(table)

Debug: Processing token 1_0 with tag None
Debug: Adding new column for tag 'None'
Debug: Adding new row for token 1_0
Debug: Incrementing 1_0 for label 'None' to 1
Debug: Processing token 1_1 with tag None
Debug: Adding new row for token 1_1
Debug: Incrementing 1_1 for label 'None' to 1
Debug: Processing token 1_2 with tag Subject
Debug: Adding new column for tag 'Subject'
Debug: Adding new row for token 1_2
Debug: Incrementing 1_2 for label 'Subject' to 1
Debug: Processing token 1_3 with tag Predicate
Debug: Adding new column for tag 'Predicate'
Debug: Adding new row for token 1_3
Debug: Incrementing 1_3 for label 'Predicate' to 1
Debug: Processing token 1_4 with tag Predicate
Debug: Adding new row for token 1_4
Debug: Incrementing 1_4 for label 'Predicate' to 1
Debug: Processing token 1_5 with tag Predicate
Debug: Adding new row for token 1_5
Debug: Incrementing 1_5 for label 'Predicate' to 1
Debug: Processing token 1_6 with tag Object
Debug: Adding new column for tag 'Object'
Debug:

In [ ]:

def fleiss_kappa(M):
    """
    Calculates the Fleiss' Kappa score, handling cases of unanimous agreement.
    
    """
    if isinstance(M, pd.DataFrame):
        M = M.values
    
    n, k = M.shape  # n is number of items, k is number of categories
    N = np.sum(M, axis=1)  # number of ratings per item
    n_bar = np.mean(N)  # mean ratings per item

    # Calculate p_j, the proportion of all ratings belonging to the j-th category
    p_j = np.sum(M, axis=0) / (n * n_bar)

    # Calculate P_i, the extent to which raters agree for the i-th item
    # Handle cases where N * (N - 1) == 0 (perfect agreement or single rating)
    P_i = (np.sum(M**2, axis=1) - N) / (N * (N - 1)) if N.size > 1 else np.zeros(n)
    P_i[np.isnan(P_i)] = 1  # Set P_i to 1 where the original calculation fails due to 0/0

    # Calculate P_bar, the mean of P_i
    P_bar = np.mean(P_i)

    # Calculate P_e, the hypothetical probability of chance agreement
    P_e = np.sum(p_j**2)

    # Handle the edge case where P_e == 1, leading to division by zero in the kappa calculation
    if P_e == 1:
        return 1.0

    kappa = (P_bar - P_e) / (1 - P_e)
    
    return kappa


# Calculate Fleiss' Kappa for SPO-labels
fleiss_kappa_spo = fleiss_kappa(table.values)
print(f"The Fleiss' Kappa score for inter-annotator agreement on SPO-labels is: {fleiss_kappa_spo:.3f}")


The Fleiss' Kappa score for inter-annotator agreement on SPO-labels is: 0.625


In [ ]:
def calculate_disagreement_percentage(contingency_matrix):
    """
    Calculate the percentage of disagreement based on a contingency matrix.
    
    """
    # Calculate the total annotations per token (sum of rows)
    total_annotations = contingency_matrix.sum(axis=1)

    # Find the maximum number of annotations per category per token (max value in each row)
    max_annotations = contingency_matrix.max(axis=1)

    # Calculate the total possible annotations (sum of all annotations)
    total_possible_annotations = total_annotations.sum()

    # Calculate the total number of annotations that match the most popular category for each token
    total_agreement_annotations = max_annotations.sum()

    # Calculate the total number of annotations that do not match the most popular category
    total_disagreement_annotations = total_possible_annotations - total_agreement_annotations

    # Calculate the percentage of disagreement
    disagreement_percentage = (total_disagreement_annotations / total_possible_annotations) * 100

    return disagreement_percentage


# Calculate and print the disagreement percentage
disagreement_percentage = calculate_disagreement_percentage(table)
print(f"Disagreement Percentage: {disagreement_percentage:.2f}%")

Disagreement Percentage: 13.74%


In [ ]:
def check_labels(contingency_matrix):
    """
    Checks if any token received three different labels from the annotators.

    """
    tokens_with_three_labels = contingency_matrix[contingency_matrix.apply(lambda x: (x > 0).sum(), axis=1) >= 3] # Filter rows where three or more columns have non-zero counts
    num_tokens_three_labels = len(tokens_with_three_labels)  # Number of tokens with three labels

    return tokens_with_three_labels, num_tokens_three_labels


# Check for tokens with three different labels
tokens_three_labels, count_three_labels = check_labels(table)
if count_three_labels > 0:
    print(f"Some tokens received three different labels: {count_three_labels} tokens")
    print(tokens_three_labels)
else:
    print("No token received three different labels.")

Some tokens received three different labels: 85 tokens
        None  Subject  Predicate  Object  Object;Subject  Predicate;Predicate  \
4_11       0        0          1       1               0                    0   
4_12       0        0          1       1               0                    0   
14_3       1        0          1       1               0                    0   
14_4       1        0          1       1               0                    0   
16_3       0        0          1       1               1                    0   
16_10      0        1          0       1               0                    0   
16_11      0        1          0       1               0                    0   
16_12      0        1          0       1               0                    0   
16_13      0        1          0       1               0                    0   
16_14      0        1          0       1               0                    0   
16_15      0        1          0       1              

In [ ]:
def majority_voting(contingency_matrix):
    """
    Performs majority voting to determine the final label for each token based on a contingency matrix.
    In cases of a complete tie with different annotations, assigns a label randomly from those that received votes.

    """
    def resolve_label(row):
        """
        Determines the final label for each token.
        
        """
        max_votes = row.max()
        # Check if there is a clear majority
        if (row == max_votes).sum() == 1:
            return row.idxmax()
        else:
            # Filter for columns that received votes and choose one randomly in case of a tie
            possible_labels = row[row > 0].index.tolist()
            return random.choice(possible_labels) if possible_labels else 'No Votes'

    # Apply the function to each row of the Df
    final_labels = contingency_matrix.apply(resolve_label, axis=1)

    return final_labels


# Perform majority voting to determine final labels
final_labels = majority_voting(table)
print(final_labels)

1_0                          None
1_1                          None
1_2                       Subject
1_3                     Predicate
1_4                     Predicate
1_5                     Predicate
1_6                        Object
1_7                        Object
1_8                        Object
1_9                        Object
1_10                       Object
1_11                         None
1_12                         None
2_0                          None
2_1                       Subject
2_2                     Predicate
2_3                        Object
2_4                        Object
2_5                        Object
2_6                          None
3_0                          None
3_1                          None
3_2                          None
3_3                          None
3_4                          None
3_5                          None
3_6                          None
3_7                          None
4_0                       Subject
4_1           

In [ ]:
data.head()

,annotator_id,sentence_id,tokens,sentence,spo_labels
0,annotator_1,1,"['P', ':', 'I', 'have', 'been', 'facing', 'som...",P : I have been facing some discomfort in my f...,"[None, None, Subject, Predicate, Predicate, Pr..."
1,annotator_1,2,"['Is', 'this', 'connected', 'to', 'my', 'diabe...",Is this connected to my diabetes ?,"[None, Subject, Predicate, Predicate, Object, ..."
2,annotator_1,3,"['A', ':', 'It', 'could', 'be', ',', 'Abdullah...","A : It could be , Abdullah .","[None, None, None, None, None, None, None, None]"
3,annotator_1,4,"['Diabetes', 'can', 'cause', 'nerve', 'damage'...",Diabetes can cause nerve damage known as perip...,"[Subject, Predicate, Predicate, Object, Object..."
4,annotator_1,5,"['This', 'can', 'lead', 'to', 'a', 'sense', 'o...","This can lead to a sense of discomfort , numbn...","[Subject, Predicate, Predicate, Object, Object..."


In [ ]:
# Creating the ready-to-use final test dataset

def update_labels(row):
    """
    Updates the initial SPO-labels with those determined by majority voting

    """
    updated_labels = []
    tokens = eval(row['tokens'])  # Assuming tokens are stored as a string representation of a list
    for idx, token in enumerate(tokens):
        key = f"{row['sentence_id']}_{idx}"
        if key in final_labels:
            updated_labels.append(final_labels[key])
        else:
            updated_labels.append("None")  # Default to 'None' if no label is provided
    return updated_labels


# Dropping 'annotator_id' and preventing duplicate sentence_id entries
data = data.drop('annotator_id', axis=1)
data = data.drop_duplicates(subset='sentence_id')

# Update the SPO-labels with those determined by majority voting
data['spo_labels'] = data.apply(update_labels, axis=1)

# Save the data
data_path = os.path.join('final data', 'final_test_data.csv')
if not os.path.exists('final data'):
    os.makedirs('final_test_data.csv')

data.to_csv(data_path, index=False)
